# Temperature and humidity forecasting with ARIMA

This version replaces the linear-regression feature-engineering approach with ARIMA models.

Main difference: ARIMA is fitted directly on the time series, so you do **not** manually create lag and rolling columns. Here we fit one ARIMA model for `temperature` and one ARIMA model for `humidity`.


In [25]:
# Uncomment if statsmodels is not installed
# %pip install statsmodels
# %pip install numpy pandas pymongo scikit-learn matplotlib


In [26]:
from itertools import product
import warnings

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


In [27]:
def get_data_from_mongodb(
    uri='mongodb://localhost:27017',
    db_name='iot_project',
    collection='measurements'
):
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=1000)
        client.admin.command('ping')
        print('Connection successful')

        db = client[db_name]
        collection = db[collection]

        cursor = collection.find({})
        df = pd.DataFrame(list(cursor))
        print(list(df.columns))
        return df
    except ConnectionFailure:
        print('Connection failed')
        return pd.DataFrame()


def clean_data(df):
    df = df.copy()

    # Delete high-correlation and irrelevant columns, ignoring columns that may not exist.
    df = df.drop(
        columns=['_id', 'node_id', 'raw_temperature', 'raw_humidity', 'raw_line', 'count'],
        errors='ignore'
    )

    # Delete rows with more than 90% missing data.
    df = df.dropna(thresh=int(0.9 * len(df.columns)), axis=0)

    # Delete duplicate rows.
    df = df.drop_duplicates()

    # Delete statistical outliers.
    for column in df.select_dtypes(include=['float64', 'int64']).columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 20 * iqr
        upper_bound = q3 + 20 * iqr
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    # Delete realistic outliers.
    df.loc[(df['temperature'] < -20) | (df['temperature'] > 60), 'temperature'] = np.nan
    df.loc[(df['humidity'] < 0) | (df['humidity'] > 100), 'humidity'] = np.nan

    # Set timestamp and sort. Your original notebook sorted but did not assign the result.
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    # Fill blanks after sorting by time.
    df[['temperature', 'humidity']] = df[['temperature', 'humidity']].interpolate().ffill().bfill()

    return df


In [28]:
raw_data = get_data_from_mongodb()

display(raw_data)
display(raw_data.describe())


Connection successful
['_id', 'timestamp', 'node_id', 'temperature', 'humidity', 'count', 'raw_temperature', 'raw_humidity', 'raw_line']


,_id,timestamp,node_id,temperature,humidity,count,raw_temperature,raw_humidity,raw_line
0,6a0b41f94d23a6034fbe0721,2026-05-18 16:44:41.444,1,26.64,45.17,294,6624,1332,"node=1,temp=6624,humidity=1332,count=294"
1,6a0b42224d23a6034fbe0722,2026-05-18 16:45:22.478,1,26.62,45.20,296,6622,1333,"node=1,temp=6622,humidity=1333,count=296"
2,6a0b42374d23a6034fbe0723,2026-05-18 16:45:43.017,1,26.60,45.23,297,6620,1334,"node=1,temp=6620,humidity=1334,count=297"
3,6a0b424c6b194568a8d38e22,2026-05-18 16:46:04.975,1,26.58,45.36,298,6618,1338,"node=1,temp=6618,humidity=1338,count=298"
4,6a0b42606b194568a8d38e23,2026-05-18 16:46:24.053,1,26.56,45.46,299,6616,1341,"node=1,temp=6616,humidity=1341,count=299"
...,...,...,...,...,...,...,...,...,...
15321,6a10d84c001f7d2d8940e2ad,2026-05-22 22:27:24.803,1,26.33,46.29,17765,6593,1367,"node=1,temp=6593,humidity=1367,count=17765"
15322,6a10d861001f7d2d8940e2ae,2026-05-22 22:27:45.321,1,26.34,46.19,17766,6594,1364,"node=1,temp=6594,humidity=1364,count=17766"
15323,6a10d875001f7d2d8940e2af,2026-05-22 22:28:05.857,1,26.37,46.23,17767,6597,1365,"node=1,temp=6597,humidity=1365,count=17767"
15324,6a10d88a001f7d2d8940e2b0,2026-05-22 22:28:26.396,1,26.38,46.23,17768,6598,1365,"node=1,temp=6598,humidity=1365,count=17768"


,timestamp,node_id,temperature,humidity,count,raw_temperature,raw_humidity
count,15326,15326.0,15326.000000,15326.000000,15326.000000,15326.000000,15326.000000
mean,2026-05-21 02:26:34.923432,1.0,26.005726,47.613185,10048.834791,6560.572556,1408.612554
min,2026-05-18 16:44:41.444000,1.0,23.920000,26.810000,0.000000,6352.000000,778.000000
25%,2026-05-20 04:55:48.269750,1.0,25.830000,47.580000,6275.250000,6543.000000,1407.000000
50%,2026-05-21 02:46:47.827000,1.0,25.930000,47.780000,10106.500000,6553.000000,1414.000000
75%,2026-05-22 00:37:47.373250,1.0,26.150000,47.960000,13937.750000,6575.000000,1419.000000
max,2026-05-22 22:28:46.919000,1.0,38.820000,53.010000,17769.000000,7842.000000,1585.000000
std,NaN,0.0,0.390387,0.959368,4535.955613,39.038729,29.883226


In [29]:
data = clean_data(raw_data)

display(data)
display(data.describe())


,timestamp,temperature,humidity
0,2026-05-18 16:44:41.444,26.64,45.17
1,2026-05-18 16:45:22.478,26.62,45.20
2,2026-05-18 16:45:43.017,26.60,45.23
3,2026-05-18 16:46:04.975,26.58,45.36
4,2026-05-18 16:46:24.053,26.56,45.46
...,...,...,...
15305,2026-05-22 22:27:24.803,26.33,46.29
15306,2026-05-22 22:27:45.321,26.34,46.19
15307,2026-05-22 22:28:05.857,26.37,46.23
15308,2026-05-22 22:28:26.396,26.38,46.23


,timestamp,temperature,humidity
count,15310,15310.000000,15310.000000
mean,2026-05-21 02:30:10.345421,25.998240,47.627576
min,2026-05-18 16:44:41.444000,23.920000,40.220000
25%,2026-05-20 04:59:54.645000,25.830000,47.580000
50%,2026-05-21 02:49:32.074500,25.930000,47.780000
75%,2026-05-22 00:39:09.495750,26.150000,47.960000
max,2026-05-22 22:28:46.919000,29.110000,53.010000
std,NaN,0.292947,0.835900


## ARIMA helper functions

Because your measurements appear to be every 20 seconds, the default frequency below is `20s`. If your real interval is different, change `freq='20s'`.


In [30]:
def prepare_arima_series(df, column, freq='20s'):
    # Return one clean, regular time series for ARIMA.
    series = (
        df[['timestamp', column]]
        .dropna(subset=['timestamp'])
        .copy()
    )
    series['timestamp'] = pd.to_datetime(series['timestamp'])
    series = series.sort_values('timestamp')

    # If several readings have the same timestamp, average them.
    series = series.groupby('timestamp')[column].mean().astype(float)

    # Resample to regular intervals, filling missing timestamps with NaN.
    if freq is not None:
        series = series.resample(freq).mean()

    # Fill missing values caused by missing sensor readings or resampling gaps.
    series = series.interpolate(method='time').ffill().bfill()
    return series


def split_series(series, train_ratio=0.8, val_ratio=0.1):
    train_end = int(train_ratio * len(series))
    val_end = int((train_ratio + val_ratio) * len(series))

    train = series.iloc[:train_end]
    val = series.iloc[train_end:val_end]
    test = series.iloc[val_end:]
    return train, val, test


def fit_arima(train, order):
    return ARIMA(
        train,
        order=order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit()


def find_best_arima(train, val):
    best = {
        'order': None,
        'model': None,
        'val_mse': np.inf,
        'val_r2': None,
    }

    for order in list(product([0, 1, 2], [0, 1], [0, 1, 2])):
        try:
            #ingore warnings if they appear
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                model = fit_arima(train, order)
                val_pred = model.forecast(steps=len(val))

            val_mse = mean_squared_error(val, val_pred)
            val_r2 = r2_score(val, val_pred)

            if val_mse < best['val_mse']:
                best = {
                    'order': order,
                    'model': model,
                    'val_mse': val_mse,
                    'val_r2': val_r2,
                }
                print(f'New best ARIMA{order}: val MSE={val_mse:.5f}, val R^2={val_r2:.5f}')
        except Exception:
            # Some ARIMA combinations can fail to converge. That is normal during a grid search.
            print(f"ARIMA with parameters: {order} not supported by ARIMA, continuing")
    return best


## Train and evaluate one ARIMA model for temperature and one for humidity


In [ ]:
FREQ = '20s'

temp_series = prepare_arima_series(data, 'temperature', freq=FREQ)
hum_series = prepare_arima_series(data, 'humidity', freq=FREQ)

print('Temperature points:', len(temp_series))
print('Humidity points:', len(hum_series))

temp_train, temp_val, temp_test = split_series(temp_series)
hum_train, hum_val, hum_test = split_series(hum_series)

print('\nSearching temperature ARIMA model')
temp_best = find_best_arima(temp_train, temp_val)

print('\nSearching humidity ARIMA model')
hum_best = find_best_arima(hum_train, hum_val)

print('\nBest temperature order:', temp_best['order'], 'validation MSE:', temp_best['val_mse'])
print('Best humidity order:', hum_best['order'], 'validation MSE:', hum_best['val_mse'])


Temperature points: 18313
Humidity points: 18313

Searching temperature ARIMA model
New best ARIMA(0, 0, 0): val MSE=0.08784, val R^2=-66.42010
New best ARIMA(0, 0, 1): val MSE=0.08782, val R^2=-66.40613
New best ARIMA(0, 0, 2): val MSE=0.08779, val R^2=-66.38048
New best ARIMA(0, 1, 0): val MSE=0.00143, val R^2=-0.09683
New best ARIMA(0, 1, 1): val MSE=0.00133, val R^2=-0.02447


In [ ]:
temp_test_result = temp_best["model"].forecast(steps=len(temp_test))
hum_test_result = hum_best["model"].forecast(steps=len(hum_test))

print('Temperature test MSE:', mean_squared_error(temp_test, temp_test_result))
print('Temperature test R^2:', r2_score(temp_test, temp_test_result))
print('Humidity test MSE:', mean_squared_error(hum_test, hum_test_result))
print('Humidity test R^2:', r2_score(hum_test, hum_test_result))


In [ ]:
instants_to_predict = 1000

# Refit final models on all available data before future forecasting.
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    final_temp_model = fit_arima(temp_series, temp_best['order'])
    final_hum_model = fit_arima(hum_series, hum_best['order'])

future_temp = final_temp_model.forecast(steps=instants_to_predict)
future_hum = final_hum_model.forecast(steps=instants_to_predict)

predictions = pd.DataFrame({
    'temperature': future_temp,
    'humidity': future_hum,
})

next_10_predictions = predictions.iloc[:10]
pred_1h_later = predictions.iloc[180]   # 180 * 20 seconds = 1 hour
pred_2h_later = predictions.iloc[360]   # 360 * 20 seconds = 2 hours
pred_5h_later = predictions.iloc[900]   # 900 * 20 seconds = 5 hours

print('Next 10 predictions:', next_10_predictions)
print('Prediction 1h later:', pred_1h_later)
print('Prediction 2h later:', pred_2h_later)
print('Prediction 5h later:', pred_5h_later)
